In [115]:
import pandas as pd
import numpy as np

In [116]:
df = pd.read_csv("../data/raw/DES_District_Data_2013_14_to_2023_24_Combined.csv")

print("Original Shape:", df.shape)

Original Shape: (101703, 37)


In [117]:
area_cols = [col for col in df.columns if col.startswith("Area-")]
year_ranges = [col.replace("Area-", "") for col in area_cols]

print("Year ranges:", year_ranges)

Year ranges: ['2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']


In [118]:
long_df_list = []

for yr_range in year_ranges:
    
    # Convert "2013-14" → 2014
    end_year = int("20" + yr_range.split("-")[1])
    
    temp = df[[
        "State",
        "District",
        "Crop",
        "Season",
        f"Area-{yr_range}",
        f"Production-{yr_range}",
        f"Yield-{yr_range}"
    ]].copy()
    
    temp.columns = [
        "State",
        "District",
        "Crop",
        "Season",
        "Area",
        "Production",
        "Yield"
    ]
    
    temp["Year"] = end_year
    
    long_df_list.append(temp)

df_long = pd.concat(long_df_list, ignore_index=True)

print("After Reshaping:", df_long.shape)
print("Unique Years:", sorted(df_long["Year"].unique()))

After Reshaping: (1118733, 8)
Unique Years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [119]:
# Remove 'Total' Rows

print("Before removing totals:", df_long.shape)

df_long = df_long[
    ~df_long["Crop"].str.contains("Total", case=False, na=False)
]

df_long = df_long[
    ~df_long["Season"].str.contains("Total", case=False, na=False)
]

print("After removing totals:", df_long.shape)

Before removing totals: (1118733, 8)
After removing totals: (594748, 8)


In [120]:
# Convert to Numeric
df_long["Area"] = pd.to_numeric(df_long["Area"], errors="coerce")
df_long["Production"] = pd.to_numeric(df_long["Production"], errors="coerce")
df_long["Yield"] = pd.to_numeric(df_long["Yield"], errors="coerce")

In [121]:
print("Before dropping NA:", df_long.shape)

df_long = df_long.dropna(subset=["Area", "Production", "Yield"])

print("After dropping NA:", df_long.shape)

Before dropping NA: (594748, 8)
After dropping NA: (133042, 8)


In [122]:
df_long = df_long[
    (df_long["Area"] > 0) &
    (df_long["Production"] > 0) &
    (df_long["Yield"] > 0)
]

print("After removing zero values:", df_long.shape)

After removing zero values: (80666, 8)


In [123]:
aggregated_crops = [
    "Cereals",
    "Nutri/Coarse Cereals",
    "Shree Anna /Nutri Cereals",
    "Other Pulses",
    "Small Millets"
]

df_long = df_long[~df_long["Crop"].isin(aggregated_crops)]

print("After removing aggregated crops:", df_long.shape)

After removing aggregated crops: (44324, 8)


In [124]:
# Convert Lakh Hectares → Hectares
df_long["Area"] = df_long["Area"] * 100000

# Convert Lakh Tonnes → Tonnes
df_long["Production"] = df_long["Production"] * 100000

In [125]:
# filter crops

real_crops = [
    'Rice', 'Maize', 'Tur', 'Urad', 'Moong',
    'Jowar', 'Bajra', 'Ragi',
    'Gram', 'Wheat', 'Lentil', 'Barley'
]

df_final = df_long[df_long['Crop'].isin(real_crops)].copy()

df_final = df_final.drop(columns=['Production'])

df_final = df_final.reset_index(drop=True)

print("Final ML Ready Shape:", df_final.shape)
print("Crops:", df_final['Crop'].unique())

Final ML Ready Shape: (44324, 7)
Crops: <StringArray>
[  'Rice',  'Maize',  'Jowar',  'Bajra',   'Ragi',    'Tur',   'Gram',
   'Urad',  'Moong',  'Wheat', 'Lentil', 'Barley']
Length: 12, dtype: str


In [126]:
print(df_final['Crop'].value_counts())

Crop
Rice      10209
Maize      7564
Wheat      4548
Urad       3826
Gram       3135
Moong      3017
Tur        2916
Jowar      2655
Bajra      2130
Lentil     1882
Barley     1414
Ragi       1028
Name: count, dtype: int64


In [128]:
print(df_final.columns)

Index(['State', 'District', 'Crop', 'Season', 'Area', 'Yield', 'Year'], dtype='str')


In [129]:
df_final.head()
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 44324 entries, 0 to 44323
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   State     44324 non-null  str    
 1   District  44324 non-null  str    
 2   Crop      44324 non-null  str    
 3   Season    44324 non-null  str    
 4   Area      44324 non-null  float64
 5   Yield     44324 non-null  float64
 6   Year      44324 non-null  int64  
dtypes: float64(2), int64(1), str(4)
memory usage: 2.4 MB


In [127]:
df_final.to_csv("../data/processed/clean_crop_yield_data.csv", index=False)
print("Saved Clean Dataset ✅")

Saved Clean Dataset ✅
